# LSTM Regime Model

This notebook contains the full LSTM workflow for the project:

1. Load stock data
2. Clean and reshape stock data
3. Create price features
4. Merge sentiment features
5. Create market-regime labels
6. Build 30-day LSTM sequences
7. Train an LSTM classifier
8. Output regime probabilities for the bandit

The LSTM predicts the next trading day's regime probabilities:

```text
[p_trending, p_meanrev, p_highvol]
```


## 1. Imports and Setup

In [29]:
from google.colab import files

# Upload the remaining files
uploaded = files.upload()

for fn in uploaded.keys():
  print(f'User uploaded file "{fn}" with length {len(uploaded[fn])} bytes')

Saving Val_Dataset_No_Sentiment.csv to Val_Dataset_No_Sentiment (3).csv
Saving Train_Dataset_No_Sentiment.csv to Train_Dataset_No_Sentiment (3).csv
User uploaded file "Val_Dataset_No_Sentiment (3).csv" with length 86307 bytes
User uploaded file "Train_Dataset_No_Sentiment (3).csv" with length 871772 bytes


In [30]:
from google.colab import files

# Upload the files
# Click 'Choose Files' and select all your CSV files (Train_Dataset_No_Sentiment.csv, Val_Dataset_No_Sentiment.csv, Test_Dataset_No_Sentiment.csv, sentiment_features.csv)
uploaded = files.upload()

for fn in uploaded.keys():
  print(f'User uploaded file "{fn}" with length {len(uploaded[fn])} bytes')


Saving Test_Dataset_No_Sentiment.csv to Test_Dataset_No_Sentiment (1).csv
User uploaded file "Test_Dataset_No_Sentiment (1).csv" with length 290504 bytes


In [31]:
from google.colab import files

# Upload the files
# Click 'Choose Files' and select all your CSV files (Train_Dataset_No_Sentiment.csv, Val_Dataset_No_Sentiment.csv, Test_Dataset_No_Sentiment.csv, sentiment_features.csv)
uploaded = files.upload()

for fn in uploaded.keys():
  print(f'User uploaded file "{fn}" with length {len(uploaded[fn])} bytes')


Saving sentiment_features.csv to sentiment_features (1).csv
User uploaded file "sentiment_features (1).csv" with length 1557768 bytes


In [32]:
from google.colab import drive
drive.mount('/content/drive')

# --- IMPORTANT: UPDATE THIS PATH ---
# Replace this with the actual path to your dataset folder in Google Drive.
# For example: '/content/drive/MyDrive/MyProject/data'
DRIVE_PATH = '/content/' # <--- Updated path for locally uploaded files!

print(f"Assuming your CSV files are in: {DRIVE_PATH}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Assuming your CSV files are in: /content/


In [33]:
import os
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix

# If torch is missing, run this once:
# !pip install torch

import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

ROOT = Path('.')
OUTPUT_DIR = ROOT / 'lstm_outputs'
OUTPUT_DIR.mkdir(exist_ok=True)

WINDOW = 30
RANDOM_SEED = 42

np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

# Update PRICE_FILES to use the DRIVE_PATH
PRICE_FILES = {
    'train': Path(DRIVE_PATH) / 'Train_Dataset_No_Sentiment.csv',
    'val': Path(DRIVE_PATH) / 'Val_Dataset_No_Sentiment.csv',
    'test': Path(DRIVE_PATH) / 'Test_Dataset_No_Sentiment.csv',
}

SENTIMENT_FILE = Path(DRIVE_PATH) / 'sentiment_features.csv'

LABEL_NAMES = ['trending', 'meanrev', 'highvol']
LABEL_TO_ID = {name: i for i, name in enumerate(LABEL_NAMES)}
ID_TO_LABEL = {i: name for i, name in enumerate(LABEL_NAMES)}

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', device)

Using device: cpu


## 2. Load and Reshape Stock Data

The stock CSV files are in a wide Yahoo Finance style format. This converts them into one row per `(date, ticker)`.


In [34]:
def read_price_file(path, split):
    raw = pd.read_csv(path, header=None)

    tickers = raw.iloc[0, 1:].astype(str).to_numpy()
    fields = raw.iloc[1, 1:].astype(str).str.lower().to_numpy()

    data = raw.iloc[3:].copy()
    data = data.rename(columns={0: 'date'})
    data['date'] = pd.to_datetime(data['date'])

    pieces = []
    for col_idx, (ticker, field) in enumerate(zip(tickers, fields), start=1):
        if field not in {'close', 'high', 'low', 'open', 'volume'}:
            continue

        piece = data[['date', col_idx]].copy()
        piece['ticker'] = ticker
        piece['field'] = field
        piece = piece.rename(columns={col_idx: 'value'})
        pieces.append(piece)

    long = pd.concat(pieces, ignore_index=True)
    long['value'] = pd.to_numeric(long['value'], errors='coerce')

    wide = (
        long.pivot_table(
            index=['date', 'ticker'],
            columns='field',
            values='value',
            aggfunc='first'
        )
        .reset_index()
        .sort_values(['ticker', 'date'])
    )

    wide['split'] = split
    return wide

price_df = pd.concat(
    [read_price_file(path, split) for split, path in PRICE_FILES.items()],
    ignore_index=True
)

price_df.head()

field,date,ticker,close,high,low,open,volume,split
0,2022-01-03,APA,24.343285,24.412688,23.406337,23.493092,9319700.0,train
1,2022-01-04,APA,25.557844,25.974267,24.811757,24.898512,13218200.0,train
2,2022-01-05,APA,24.638248,26.104398,24.612223,26.026319,9212700.0,train
3,2022-01-06,APA,25.696651,25.948238,24.967914,25.471090,7626800.0,train
4,2022-01-07,APA,25.740030,26.286582,25.523144,25.896188,8355400.0,train


Now that the files are uploaded, we need to update the `DRIVE_PATH` to point to the Colab's local content directory. Please make sure that all the necessary CSV files are uploaded into this directory before proceeding.

The `FileNotFoundError` you encountered was because the notebook couldn't find the CSV files in the current working directory. With Google Drive mounted and the `DRIVE_PATH` updated, the code should now be able to locate your datasets.

## 3. Keep Stocks with Sentiment Data

The sentiment file covers these tickers:

```text
XOM, CVX, COP, OXY, PXD, SLB, HAL, EOG, DVN, MPC
```

The stock files include a few more tickers, so this keeps only tickers that have sentiment coverage.


In [35]:
sentiment_raw = pd.read_csv(SENTIMENT_FILE)
sentiment_tickers = sorted(sentiment_raw['ticker'].unique())

price_df = price_df[price_df['ticker'].isin(sentiment_tickers)].copy()

print('Tickers used:', sorted(price_df['ticker'].unique()))
print('Rows:', len(price_df))
price_df.head()

Tickers used: ['COP', 'CVX', 'DVN', 'EOG', 'HAL', 'MPC', 'OXY', 'SLB', 'XOM']
Rows: 9684


field,date,ticker,close,high,low,open,volume,split
752,2022-01-03,COP,63.208145,63.370944,61.708697,61.717268,5769900.0,train
753,2022-01-04,COP,65.949982,66.301277,63.645115,63.885025,9189300.0,train
754,2022-01-05,COP,64.818977,67.063860,64.707586,66.815386,9034500.0,train
755,2022-01-06,COP,67.252373,67.509424,65.890020,66.678300,8679000.0,train
756,2022-01-07,COP,69.094551,69.343032,67.158120,67.577970,10838800.0,train


## 4. Create Price Features

For each stock and day, we create:

- daily log return
- 5, 10, and 20 day returns
- moving-average ratios
- rolling volatility
- momentum
- RSI
- volume z-score
- cumulative return
- annualized volatility
- Sharpe ratio
- maximum drawdown
- win rate


In [36]:
def rsi(close, period=14):
    delta = close.diff()
    gain = delta.clip(lower=0).rolling(period, min_periods=period).mean()
    loss = (-delta.clip(upper=0)).rolling(period, min_periods=period).mean()
    rs = gain / loss.replace(0, np.nan)
    return 100 - (100 / (1 + rs))


def rolling_max_drawdown(close, window=20):
    def calc(values):
        running_peak = np.maximum.accumulate(values)
        drawdowns = values / running_peak - 1
        return float(drawdowns.min())

    return close.rolling(window, min_periods=window).apply(calc, raw=True)


def add_price_features(df):
    out = df.sort_values(['ticker', 'date']).copy()
    groups = out.groupby('ticker', group_keys=False)

    out['log_return'] = groups['close'].transform(lambda s: np.log(s / s.shift(1)))
    out['return_5d'] = groups['close'].transform(lambda s: np.log(s / s.shift(5)))
    out['return_10d'] = groups['close'].transform(lambda s: np.log(s / s.shift(10)))
    out['return_20d'] = groups['close'].transform(lambda s: np.log(s / s.shift(20)))

    for window in [5, 10, 20]:
        ma = groups['close'].transform(lambda s, w=window: s.rolling(w, min_periods=w).mean())
        out[f'ma_{window}_ratio'] = out['close'] / ma - 1

    out['volatility_10d'] = groups['log_return'].transform(lambda s: s.rolling(10, min_periods=10).std())
    out['volatility_20d'] = groups['log_return'].transform(lambda s: s.rolling(20, min_periods=20).std())
    out['momentum_10d'] = groups['log_return'].transform(lambda s: s.rolling(10, min_periods=10).sum())
    out['rsi_14'] = groups['close'].transform(rsi)

    volume_mean = groups['volume'].transform(lambda s: s.rolling(20, min_periods=20).mean())
    volume_std = groups['volume'].transform(lambda s: s.rolling(20, min_periods=20).std())
    out['volume_z_20d'] = (out['volume'] - volume_mean) / volume_std.replace(0, np.nan)

    out['cumulative_return_20d'] = out['return_20d']
    out['annualized_volatility_20d'] = out['volatility_20d'] * np.sqrt(252)

    rolling_mean = groups['log_return'].transform(lambda s: s.rolling(20, min_periods=20).mean())
    rolling_std = groups['log_return'].transform(lambda s: s.rolling(20, min_periods=20).std())
    out['sharpe_20d'] = (rolling_mean / rolling_std.replace(0, np.nan)) * np.sqrt(252)

    out['max_drawdown_20d'] = groups['close'].transform(rolling_max_drawdown)
    out['win_rate_20d'] = groups['log_return'].transform(lambda s: (s > 0).rolling(20, min_periods=20).mean())

    return out

price_features_df = add_price_features(price_df)
price_features_df.head()

field,date,ticker,close,high,low,open,volume,split,log_return,return_5d,...,volatility_10d,volatility_20d,momentum_10d,rsi_14,volume_z_20d,cumulative_return_20d,annualized_volatility_20d,sharpe_20d,max_drawdown_20d,win_rate_20d
752,2022-01-03,COP,63.208145,63.370944,61.708697,61.717268,5769900.0,train,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
753,2022-01-04,COP,65.949982,66.301277,63.645115,63.885025,9189300.0,train,0.042463,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
754,2022-01-05,COP,64.818977,67.063860,64.707586,66.815386,9034500.0,train,-0.017298,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
755,2022-01-06,COP,67.252373,67.509424,65.890020,66.678300,8679000.0,train,0.036854,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
756,2022-01-07,COP,69.094551,69.343032,67.158120,67.577970,10838800.0,train,0.027024,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [37]:
PRICE_FEATURES = [
    'log_return',
    'return_5d',
    'return_10d',
    'return_20d',
    'ma_5_ratio',
    'ma_10_ratio',
    'ma_20_ratio',
    'volatility_10d',
    'volatility_20d',
    'momentum_10d',
    'rsi_14',
    'volume_z_20d',
    'cumulative_return_20d',
    'annualized_volatility_20d',
    'sharpe_20d',
    'max_drawdown_20d',
    'win_rate_20d',
]

SENTIMENT_FEATURES = [
    'p_positive_mean',
    'p_negative_mean',
    'p_neutral_mean',
    'net_sentiment',
    'sentiment_volume',
    'sentiment_momentum',
    'sentiment_dispersion',
]

FEATURE_COLS = PRICE_FEATURES + SENTIMENT_FEATURES

print('Number of LSTM features per day:', len(FEATURE_COLS))
FEATURE_COLS

Number of LSTM features per day: 24


['log_return',
 'return_5d',
 'return_10d',
 'return_20d',
 'ma_5_ratio',
 'ma_10_ratio',
 'ma_20_ratio',
 'volatility_10d',
 'volatility_20d',
 'momentum_10d',
 'rsi_14',
 'volume_z_20d',
 'cumulative_return_20d',
 'annualized_volatility_20d',
 'sharpe_20d',
 'max_drawdown_20d',
 'win_rate_20d',
 'p_positive_mean',
 'p_negative_mean',
 'p_neutral_mean',
 'net_sentiment',
 'sentiment_volume',
 'sentiment_momentum',
 'sentiment_dispersion']

## 5. Merge Sentiment Features

Sentiment is joined by `(date, ticker)`. Missing sentiment values are forward-filled by ticker, then remaining missing values become zero.


In [38]:
sentiment_df = sentiment_raw.rename(columns={'trading_day': 'date'}).copy()
sentiment_df['date'] = pd.to_datetime(sentiment_df['date'])

merged_df = price_features_df.merge(sentiment_df, on=['date', 'ticker'], how='left')
merged_df[SENTIMENT_FEATURES] = (
    merged_df
    .groupby('ticker')[SENTIMENT_FEATURES]
    .ffill()
    .fillna(0.0)
)

merged_df = (
    merged_df
    .dropna(subset=FEATURE_COLS)
    .sort_values(['ticker', 'date'])
    .reset_index(drop=True)
)

merged_df.head()

,date,ticker,close,high,low,open,volume,split,log_return,return_5d,...,sharpe_20d,max_drawdown_20d,win_rate_20d,p_positive_mean,p_negative_mean,p_neutral_mean,net_sentiment,sentiment_volume,sentiment_momentum,sentiment_dispersion
0,2022-02-01,COP,78.433968,78.733856,74.732474,74.886703,9424900.0,train,0.032418,0.049604,...,7.346349,-0.054736,0.6,-0.338670,-0.641277,0.881383,0.201442,0.000000,0.045127,-0.390029
1,2022-02-02,COP,79.008041,79.350774,76.240493,77.500024,8803400.0,train,0.007293,0.057471,...,6.488655,-0.054736,0.6,-0.338670,-0.641277,0.881383,0.201442,0.000000,0.045127,-0.390029
2,2022-02-03,COP,77.859901,79.642097,76.660342,79.496438,13258500.0,train,-0.014639,0.013405,...,6.632010,-0.054736,0.6,-0.338670,-0.641277,0.881383,0.201442,0.000000,0.045127,-0.390029
3,2022-02-04,COP,78.716736,81.338626,78.613914,78.828121,12035300.0,train,0.010945,0.029269,...,5.960304,-0.054736,0.6,-0.796347,2.051008,-1.197901,-1.743046,0.730489,-0.830994,0.111216
4,2022-02-07,COP,79.633530,80.738838,76.840271,77.474321,8420600.0,train,0.011579,0.047597,...,5.497013,-0.054736,0.6,-0.796347,2.051008,-1.197901,-1.743046,0.000000,-0.830994,0.111216


## 6. Create Regime Labels

The regime labels use past/current data only:

- `highvol`: 20-day annualized volatility is high
- `trending`: absolute 20-day cumulative return is large
- `meanrev`: everything else

The LSTM input window ending at day `t` predicts the regime for the next trading day `t+1`.


In [39]:
train_only = merged_df[merged_df['split'] == 'train']

highvol_threshold = train_only['annualized_volatility_20d'].quantile(0.75)
trend_threshold = train_only['cumulative_return_20d'].abs().quantile(0.67)

print('High-vol threshold:', highvol_threshold)
print('Trend threshold:', trend_threshold)

highvol = merged_df['annualized_volatility_20d'] >= highvol_threshold
trending = merged_df['cumulative_return_20d'].abs() >= trend_threshold

merged_df['regime'] = np.select(
    [highvol, trending],
    [LABEL_TO_ID['highvol'], LABEL_TO_ID['trending']],
    default=LABEL_TO_ID['meanrev']
).astype(int)

merged_df['regime_name'] = merged_df['regime'].map(ID_TO_LABEL)

# Predict next trading day's regime.
merged_df['target_regime'] = merged_df.groupby('ticker')['regime'].shift(-1)
merged_df['target_date'] = merged_df.groupby('ticker')['date'].shift(-1)
merged_df['target_regime_name'] = merged_df['target_regime'].map(ID_TO_LABEL)

merged_df = merged_df.dropna(subset=['target_regime']).reset_index(drop=True)
merged_df['target_regime'] = merged_df['target_regime'].astype(int)

merged_df[['date', 'ticker', 'split', 'regime_name', 'target_date', 'target_regime_name']].head()

High-vol threshold: 0.38878247380708764
Trend threshold: 0.08350375874339698


,date,ticker,split,regime_name,target_date,target_regime_name
0,2022-02-01,COP,train,trending,2022-02-02,trending
1,2022-02-02,COP,train,trending,2022-02-03,trending
2,2022-02-03,COP,train,trending,2022-02-04,trending
3,2022-02-04,COP,train,trending,2022-02-07,trending
4,2022-02-07,COP,train,trending,2022-02-08,trending


In [40]:
merged_df.groupby(['split', 'target_regime_name']).size().unstack(fill_value=0)

target_regime_name,highvol,meanrev,trending
split,,,
test,348,1594,299
train,1644,3653,1291
val,126,241,299


## 7. Scale Features

The scaler is fit on the training split only. Validation and test data use the training scaler.


In [41]:
scaler = StandardScaler()
scaled_df = merged_df.copy()

train_mask = scaled_df['split'] == 'train'
scaler.fit(scaled_df.loc[train_mask, FEATURE_COLS])
scaled_df[FEATURE_COLS] = scaler.transform(scaled_df[FEATURE_COLS])

scaler_df = pd.DataFrame({
    'feature': FEATURE_COLS,
    'mean': scaler.mean_,
    'scale': scaler.scale_,
})

scaler_df.to_csv(OUTPUT_DIR / 'lstm_feature_scaler.csv', index=False)
merged_df.to_csv(OUTPUT_DIR / 'merged_price_sentiment_regimes.csv', index=False)

scaler_df.head()

,feature,mean,scale
0,log_return,0.000261,0.021572
1,return_5d,0.001289,0.048982
2,return_10d,0.002883,0.067399
3,return_20d,0.007815,0.094623
4,ma_5_ratio,0.000610,0.023846


## 8. Create 30-Day LSTM Sequences

For each ticker, each sample is:

```text
[x_{t-29}, x_{t-28}, ..., x_t] -> regime at t+1
```


In [42]:
def make_sequences(df, split, window=30):
    split_df = df[df['split'] == split].sort_values(['ticker', 'date']).copy()

    xs = []
    ys = []
    meta_rows = []

    for ticker, grp in split_df.groupby('ticker'):
        grp = grp.reset_index(drop=True)
        values = grp[FEATURE_COLS].to_numpy(dtype=np.float32)
        targets = grp['target_regime'].to_numpy()

        for end in range(window - 1, len(grp) - 1):
            target = targets[end]
            if pd.isna(target):
                continue

            xs.append(values[end - window + 1:end + 1])
            ys.append(int(target))
            meta_rows.append({
                'ticker': ticker,
                'window_start_date': grp.loc[end - window + 1, 'date'],
                'window_end_date': grp.loc[end, 'date'],
                'target_date': grp.loc[end, 'target_date'],
                'target_regime': int(target),
                'target_regime_name': ID_TO_LABEL[int(target)],
            })

    return np.stack(xs), np.asarray(ys, dtype=np.int64), pd.DataFrame(meta_rows)

X_train, y_train, meta_train = make_sequences(scaled_df, 'train', WINDOW)
X_val, y_val, meta_val = make_sequences(scaled_df, 'val', WINDOW)
X_test, y_test, meta_test = make_sequences(scaled_df, 'test', WINDOW)

np.save(OUTPUT_DIR / 'X_train.npy', X_train)
np.save(OUTPUT_DIR / 'y_train.npy', y_train)
np.save(OUTPUT_DIR / 'X_val.npy', X_val)
np.save(OUTPUT_DIR / 'y_val.npy', y_val)
np.save(OUTPUT_DIR / 'X_test.npy', X_test)
np.save(OUTPUT_DIR / 'y_test.npy', y_test)

meta_train.to_csv(OUTPUT_DIR / 'meta_train.csv', index=False)
meta_val.to_csv(OUTPUT_DIR / 'meta_val.csv', index=False)
meta_test.to_csv(OUTPUT_DIR / 'meta_test.csv', index=False)

print('Train:', X_train.shape, y_train.shape)
print('Val:  ', X_val.shape, y_val.shape)
print('Test: ', X_test.shape, y_test.shape)

Train: (6318, 30, 24) (6318,)
Val:   (396, 30, 24) (396,)
Test:  (1971, 30, 24) (1971,)


## 9. Build the LSTM Model

The model reads a 30-day sequence with 24 features per day and outputs 3 logits. Softmax converts those logits into probabilities.


In [43]:
class RegimeLSTM(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.25, num_classes=3):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout,
        )

        self.classifier = nn.Sequential(
            nn.LayerNorm(hidden_size),
            nn.Linear(hidden_size, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, num_classes),
        )

    def forward(self, x):
        _, (hidden, _) = self.lstm(x)
        last_hidden = hidden[-1]
        logits = self.classifier(last_hidden)
        return logits

model = RegimeLSTM(input_size=X_train.shape[-1]).to(device)
model

RegimeLSTM(
  (lstm): LSTM(24, 64, num_layers=2, batch_first=True, dropout=0.25)
  (classifier): Sequential(
    (0): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
    (1): Linear(in_features=64, out_features=32, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.25, inplace=False)
    (4): Linear(in_features=32, out_features=3, bias=True)
  )
)

## 10. Train the LSTM

In [44]:
def train_model(
    model,
    X_train,
    y_train,
    X_val,
    y_val,
    epochs=30,
    batch_size=64,
    lr=1e-3,
):
    train_dataset = TensorDataset(
        torch.tensor(X_train, dtype=torch.float32),
        torch.tensor(y_train, dtype=torch.long),
    )
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    X_val_tensor = torch.tensor(X_val, dtype=torch.float32).to(device)
    y_val_tensor = torch.tensor(y_val, dtype=torch.long).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss()

    history = []
    best_val_loss = float('inf')
    best_state = None

    for epoch in range(1, epochs + 1):
        model.train()
        train_losses = []

        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)

            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            train_losses.append(loss.item())

        model.eval()
        with torch.no_grad():
            val_logits = model(X_val_tensor)
            val_loss = criterion(val_logits, y_val_tensor).item()
            val_pred = val_logits.argmax(dim=1)
            val_acc = (val_pred == y_val_tensor).float().mean().item()

        avg_train_loss = float(np.mean(train_losses))
        history.append({
            'epoch': epoch,
            'train_loss': avg_train_loss,
            'val_loss': val_loss,
            'val_acc': val_acc,
        })

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        print(f'Epoch {epoch:02d} | train_loss={avg_train_loss:.4f} | val_loss={val_loss:.4f} | val_acc={val_acc:.3f}')

    if best_state is not None:
        model.load_state_dict(best_state)

    return pd.DataFrame(history)

history = train_model(
    model,
    X_train,
    y_train,
    X_val,
    y_val,
    epochs=30,
    batch_size=64,
    lr=1e-3,
)

history.to_csv(OUTPUT_DIR / 'training_history.csv', index=False)
history.tail()

Epoch 01 | train_loss=0.5628 | val_loss=0.5345 | val_acc=0.793
Epoch 02 | train_loss=0.3754 | val_loss=0.5423 | val_acc=0.770
Epoch 03 | train_loss=0.3307 | val_loss=0.4733 | val_acc=0.798
Epoch 04 | train_loss=0.3084 | val_loss=0.4691 | val_acc=0.816
Epoch 05 | train_loss=0.2941 | val_loss=0.6283 | val_acc=0.750
Epoch 06 | train_loss=0.2864 | val_loss=0.5222 | val_acc=0.806
Epoch 07 | train_loss=0.2690 | val_loss=0.5425 | val_acc=0.806
Epoch 08 | train_loss=0.2647 | val_loss=0.6691 | val_acc=0.770
Epoch 09 | train_loss=0.2588 | val_loss=0.6430 | val_acc=0.758
Epoch 10 | train_loss=0.2552 | val_loss=0.6615 | val_acc=0.795
Epoch 11 | train_loss=0.2521 | val_loss=0.6406 | val_acc=0.773
Epoch 12 | train_loss=0.2386 | val_loss=0.5843 | val_acc=0.798
Epoch 13 | train_loss=0.2355 | val_loss=0.6495 | val_acc=0.795
Epoch 14 | train_loss=0.2250 | val_loss=0.6620 | val_acc=0.785
Epoch 15 | train_loss=0.2259 | val_loss=0.6448 | val_acc=0.788
Epoch 16 | train_loss=0.2171 | val_loss=0.5882 | val_ac

,epoch,train_loss,val_loss,val_acc
25,26,0.177540,0.852248,0.765152
26,27,0.177747,0.646726,0.800505
27,28,0.175635,0.809485,0.760101
28,29,0.162508,0.940797,0.775253
29,30,0.162265,0.846860,0.739899


## 11. Evaluate on Test Data

In [45]:
model.eval()

with torch.no_grad():
    X_test_tensor = torch.tensor(X_test, dtype=torch.float32).to(device)
    test_logits = model(X_test_tensor)
    test_probs = torch.softmax(test_logits, dim=1).cpu().numpy()
    test_pred = test_probs.argmax(axis=1)

print(classification_report(y_test, test_pred, target_names=LABEL_NAMES, zero_division=0))
confusion_matrix(y_test, test_pred)

              precision    recall  f1-score   support

    trending       0.64      0.65      0.64       227
     meanrev       0.94      0.92      0.93      1400
     highvol       0.83      0.90      0.86       344

    accuracy                           0.88      1971
   macro avg       0.80      0.82      0.81      1971
weighted avg       0.88      0.88      0.88      1971



array([[ 147,   59,   21],
       [  77, 1281,   42],
       [   7,   28,  309]])

## 12. Save Regime Probability Output

This is the output the bandit should use. Merge it into the MAB pipeline by `ticker` and `target_date`.


In [46]:
probability_output = meta_test.copy()

for i, label in enumerate(LABEL_NAMES):
    probability_output[f'p_{label}'] = test_probs[:, i]

probability_output['predicted_regime'] = test_pred
probability_output['predicted_regime_name'] = [ID_TO_LABEL[i] for i in test_pred]

probability_output.to_csv(OUTPUT_DIR / 'test_regime_probabilities.csv', index=False)
torch.save(model.state_dict(), OUTPUT_DIR / 'regime_lstm.pt')

probability_output.head()

,ticker,window_start_date,window_end_date,target_date,target_regime,target_regime_name,p_trending,p_meanrev,p_highvol,predicted_regime,predicted_regime_name
0,COP,2025-01-02,2025-02-14,2025-02-18,1,meanrev,0.264953,0.732031,0.003016,1,meanrev
1,COP,2025-01-03,2025-02-18,2025-02-19,1,meanrev,0.077621,0.921006,0.001373,1,meanrev
2,COP,2025-01-06,2025-02-19,2025-02-20,1,meanrev,0.015608,0.983814,0.000578,1,meanrev
3,COP,2025-01-07,2025-02-20,2025-02-21,1,meanrev,0.009165,0.990341,0.000495,1,meanrev
4,COP,2025-01-08,2025-02-21,2025-02-24,1,meanrev,0.024739,0.973799,0.001462,1,meanrev


In [47]:
print('Saved files:')
for path in sorted(OUTPUT_DIR.iterdir()):
    print(path)

Saved files:
lstm_outputs/X_test.npy
lstm_outputs/X_train.npy
lstm_outputs/X_val.npy
lstm_outputs/lstm_feature_scaler.csv
lstm_outputs/merged_price_sentiment_regimes.csv
lstm_outputs/meta_test.csv
lstm_outputs/meta_train.csv
lstm_outputs/meta_val.csv
lstm_outputs/regime_lstm.pt
lstm_outputs/test_regime_probabilities.csv
lstm_outputs/training_history.csv
lstm_outputs/y_test.npy
lstm_outputs/y_train.npy
lstm_outputs/y_val.npy


In [48]:
from google.colab import files

files.download("lstm_outputs/test_regime_probabilities.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>